Agatha Majcher  

# Carbon Tax Scenario Analysis — PyPSA-RSA
## Paper 0: 2030 Snapshot — 4 CT Scenarios

| Scenario | CT in opt? | Revenue recycling? |
|----------|------------|-------------------|
| P0_BASE | No | No |
| P0_BASE_R | No | Yes (CT revenues from BASE reinvested in RE) |
| P0_CT | Yes (462 R/tCO₂) | No |
| P0_CT_R | Yes | Yes |

**Network**: 10 Eskom supply regions, LC-182h (test), `build_year` multi-invest [2025, 2030].  
**Analysis year**: 2030 unless stated.

# 1. Setup

In [ ]:
python -m pip install ipykernel -U

: 

In [ ]:
conda activate pypsa

: 

In [ ]:
import pypsa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')
plt.style.use('bmh')
%matplotlib inline
from pypsa.descriptors import get_switchable_as_dense as get_as_dense

: 

In [ ]:
RESULTS = '/beegfs/scratch/agma/pypsa-rsa/results/Coal_Flexibilisation'
SCENARIOS = ['P0_BASE', 'P0_BASE_R', 'P0_CT', 'P0_CT_R']
LABELS = {'P0_BASE':'BASE', 'P0_BASE_R':'BASE+R', 'P0_CT':'CT', 'P0_CT_R':'CT+R'}
COLORS = {'P0_BASE':'#1f77b4', 'P0_BASE_R':'#ff7f0e', 'P0_CT':'#2ca02c', 'P0_CT_R':'#d62728'}

CT_RATE = 462        # R/tCO2 — official SA 2030 headline rate
SCALE_COSTS = 1e3    # scale_costs(n,1e3) divides all costs by 1e3 before solving

RE_CARRIERS = ['solar_pv','solar_pv_low','solar_pv_rooftop','wind','wind_low',
               'hydro','hydro_import','solar_csp','bioenergy']
FOSSIL_CARRIERS = ['coal','ccgt_steam','ocgt_diesel','ocgt_gas',
                   'ocgt_gas_h2_40','ocgt_gas_h2_45','ocgt_gas_h2_50',
                   'ocgt_gas_h2_55','ocgt_gas_h2_60','rmippp','sasol_coal','sasol_gas']
RE_INVEST_CARRIERS = ['wind','wind_low','solar_pv','solar_pv_low']  # in CT constraint

CARRIER_COLORS = {
    'coal':'#333333', 'nuclear':'#ff8c00', 'hydro':'#4169e1',
    'hydro_import':'#1a6bb5', 'solar_pv':'#ffd700', 'solar_pv_low':'#ffc200',
    'solar_pv_rooftop':'#e8b400', 'solar_csp':'#ff6600', 'wind':'#27ae60',
    'wind_low':'#1a7d44', 'bioenergy':'#8b4513', 'ccgt_steam':'#b0b0b0',
    'ocgt_gas':'#909090', 'ocgt_diesel':'#707070', 'sasol_coal':'#555555',
    'sasol_gas':'#aaaaaa', 'rmippp':'#cd853f', 'load_shedding':'#ff0000',
    'phs':'#00ced1', 'battery_4h':'#9b59b6', 'battery_1h':'#8e44ad', 'battery_8h':'#7d3c98',
}

In [ ]:
networks = {}
for s in SCENARIOS:
    networks[s] = pypsa.Network(f'{RESULTS}/{s}/networks/solved.nc')
    print(f'Loaded {s}: {len(networks[s].generators)} generators, '
          f'{len(networks[s].storage_units)} storage units, '
          f'{len(networks[s].links)} links')

n_base, n_base_r, n_ct, n_ct_r = [networks[s] for s in SCENARIOS]

## Helper Functions

In [ ]:
def get_2030(n):
    """Return (snapshots_2030, weights_2030) for generators."""
    sns = n.snapshots[n.snapshots.get_level_values('period') == 2030]
    w   = n.snapshot_weightings.loc[sns, 'generators']
    return sns, w

def dispatch_twh(n, year=2030):
    """Annual generation in TWh per generator (MultiIndex weighted)."""
    sns, w = get_2030(n)
    return n.generators_t.p.loc[sns].multiply(w, axis=0).sum() / 1e6

def dispatch_by_carrier(n, year=2030):
    d = dispatch_twh(n, year)
    return d.groupby(n.generators.carrier).sum()

def emissions_mt(scen, year=2030):
    """CO2 emissions in MtCO2 using generator_emissions.csv."""
    n  = networks[scen]
    em = pd.read_csv(f'{RESULTS}/{scen}/outputs/generator_emissions.csv', index_col=0)
    sns, w = get_2030(n)
    gen_mwh = n.generators_t.p.loc[sns].multiply(w, axis=0).sum()
    common  = em.columns.intersection(gen_mwh.index)
    em_kg   = (gen_mwh[common] * em.loc[year, common]).sum()
    return em_kg / 1e9  # MtCO2 (1 Mt = 1e9 kg)

def ct_revenue_bn(scen, year=2030):
    """CT revenue in bn ZAR at 462 R/tCO2."""
    return emissions_mt(scen, year) * 1e6 * CT_RATE / 1e9

def new_build_gw(n, year=2030):
    """New build capacity in GW by carrier for build_year==year."""
    nb = n.generators[n.generators.build_year == year]
    return nb.groupby('carrier').p_nom_opt.sum() / 1e3

def reinvestment_bn(n, year=2030):
    """Annualised RE capital investment in bn ZAR.
    Multiply by SCALE_COSTS because capital_cost was divided by 1e3 before solving."""
    re = n.generators[
        (n.generators.build_year == year) &
        n.generators.carrier.isin(RE_INVEST_CARRIERS)
    ]
    return (re.p_nom_opt * re.capital_cost).sum() * SCALE_COSTS / 1e9

def total_load_twh(n):
    sns, w = get_2030(n)
    return n.loads_t.p_set.loc[sns].multiply(w, axis=0).sum().sum() / 1e6

# 2. Network Sanity Check

In [ ]:
for s, n in networks.items():
    sns_2030, w_2030 = get_2030(n)
    print(f'{s}:')
    print(f'  investment_periods = {n.investment_periods.tolist()}')
    print(f'  2030 snapshots = {len(sns_2030)}, weighting sum = {w_2030.sum():.0f} h')
    print(f'  total load 2030 = {total_load_twh(n):.1f} TWh')
    print()

In [ ]:
# Components
for c in n_base.iterate_components(list(n_base.components.keys())[2:]):
    if len(c.df) > 0:
        print(f"  {c.name}: {len(c.df)} entries")

In [ ]:
# Carriers and their CO2 factors
# Note: co2_emissions in n.carriers are all 0 — emissions are in generator_emissions.csv
n_base.carriers[['co2_emissions']]

## 2.1 Loads

In [ ]:
n_base.loads.head()

In [ ]:
# 2030 load time series (BASE scenario)
sns, w = get_2030(n_base)
n_base.loads_t.p_set.loc[sns].sum(axis=1).plot(figsize=(15,3), title='2030 Total Load — P0_BASE [MW]')
plt.ylabel('MW'); plt.tight_layout();

In [ ]:
# Annual load per bus (2030)
for s, n in networks.items():
    load = total_load_twh(n)
    print(f'{s}: {load:.1f} TWh')

## 2.2 Generators

In [ ]:
n_base.generators[['carrier','bus','p_nom','p_nom_opt','build_year','marginal_cost','capital_cost']].head(15)

In [ ]:
# p_nom_opt by carrier (P0_BASE, all build years)
cap = n_base.generators.groupby('carrier').p_nom_opt.sum().div(1e3).sort_values(ascending=False)
cap[cap > 0.001].round(2)

In [ ]:
# 2030 dispatch time series (selected carriers, BASE)
sns, w = get_2030(n_base)
dispatch = n_base.generators_t.p.loc[sns]
coal_cols = [c for c in dispatch.columns if n_base.generators.loc[c, 'carrier'] == 'coal']
solar_cols = [c for c in dispatch.columns if 'solar_pv' in n_base.generators.loc[c, 'carrier']]

fig, axes = plt.subplots(2, 1, figsize=(15, 6))
dispatch[coal_cols].sum(axis=1).plot(ax=axes[0], title='Coal dispatch 2030 [MW]')
dispatch[solar_cols].sum(axis=1).plot(ax=axes[1], title='Solar PV dispatch 2030 [MW]')
plt.tight_layout();

In [ ]:
# Capacity factors 2030 (P0_BASE)
sns, w = get_2030(n_base)
p_avg = n_base.generators_t.p.loc[sns].mean()
cf = p_avg / n_base.generators.p_nom_opt.replace(0, np.nan)
cf_by_carrier = cf.groupby(n_base.generators.carrier).mean().dropna().sort_values(ascending=False)
cf_by_carrier.round(3)

In [ ]:
# p_max_pu profiles (2030, BASE)
if not n_base.generators_t.p_max_pu.empty:
    sns, _ = get_2030(n_base)
    profiles = n_base.generators_t.p_max_pu.loc[sns]
    carrier_mean = profiles.groupby(n_base.generators.carrier, axis=1).mean()
    carrier_mean.plot(figsize=(15,4), title='Mean p_max_pu profiles 2030 by carrier')
    plt.ylabel('p.u.'); plt.tight_layout();

## 2.3 Carriers

> **Note**: `n.carriers.co2_emissions` are all 0 in this model. Actual emission factors are per-generator in `outputs/generator_emissions.csv` (kgCO₂/MWh_el, period-indexed).

In [ ]:
n_base.carriers

In [ ]:
# Sample emission factors from CSV
import pandas as pd
em = pd.read_csv(f'{RESULTS}/P0_BASE/outputs/generator_emissions.csv', index_col=0)
print('Shape:', em.shape, '  (rows=periods, cols=generators)')
print('Non-zero generators (2030, first 10):')
ef_2030 = em.loc[2030]
print(ef_2030[ef_2030 > 0].head(10).round(1))

## 2.4 Storage

In [ ]:
n_base.storage_units[['carrier','bus','p_nom','p_nom_opt','build_year']].head(15)

In [ ]:
print('Storage carriers:', n_base.storage_units.carrier.unique().tolist())
print()
print('Optimal storage capacities by carrier [GW]:')
print(n_base.storage_units.groupby('carrier').p_nom_opt.sum().div(1e3).round(2))

In [ ]:
# State of charge 2030 (BASE)
sns, _ = get_2030(n_base)
fig, ax = plt.subplots(figsize=(15, 4))
n_base.storage_units_t.state_of_charge.loc[sns].sum(axis=1).div(1e3).plot(
    ax=ax, title='Total storage state of charge 2030 — P0_BASE [GWh]')
ax.set_ylabel('GWh'); plt.tight_layout();

In [ ]:
# PHS vs battery
sns, _ = get_2030(n_base)
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
for ax, carr, label in zip(axes, ['phs','battery_4h'], ['PHS','Battery 4h']):
    cols = n_base.storage_units.index[n_base.storage_units.carrier == carr]
    soc = n_base.storage_units_t.state_of_charge.loc[sns, cols].sum(axis=1).div(1e3)
    if not soc.empty:
        soc.plot(ax=ax, title=f'{label} SOC 2030 [GWh]')
        ax.set_ylabel('GWh')
plt.tight_layout();

## 2.5 Transmission (Links)

> This model uses **Links** (not Lines) for inter-regional transmission (DC-linearised).

In [ ]:
n_base.links[['bus0','bus1','p_nom','p_nom_opt','build_year','capital_cost']].head(10)

In [ ]:
# Transmission expansion (all links have build_year=2024)
expanded = n_base.links[n_base.links.p_nom_opt > n_base.links.p_nom * 1.01]
print(f'Expanded links: {len(expanded)} of {len(n_base.links)}')
expansion = (n_base.links.p_nom_opt - n_base.links.p_nom).sort_values(ascending=False)
print(expansion.head(10).round(0))

In [ ]:
# Link flows 2030 (BASE)
sns, _ = get_2030(n_base)
if not n_base.links_t.p0.empty:
    link_flows = n_base.links_t.p0.loc[sns]
    link_avg = link_flows.abs().mean().sort_values(ascending=False)
    print('Mean absolute flow 2030 [MW]:')
    print(link_avg.head(10).round(0))

## 2.6 Buses

In [ ]:
n_base.buses

---
# 3. Paper Results

All values for **2030**. Raw test runs (LC-182h, regions=10). Final paper numbers need LC (8760h) runs.

## 3.1 New Build Capacity 2030 [GW]

In [ ]:
nb_all = pd.DataFrame({s: new_build_gw(networks[s]) for s in SCENARIOS}).fillna(0)
nb_all = nb_all[nb_all.max(axis=1) > 0.01].round(2)
nb_all.index.name = 'carrier'
nb_all

In [ ]:
# Bar chart: new build by scenario
fig, ax = plt.subplots(figsize=(12, 5))
nb_plot = nb_all.drop('bioenergy', errors='ignore').T  # scenarios as rows
nb_plot.index = [LABELS[s] for s in nb_plot.index]
nb_plot.plot.bar(ax=ax, rot=0,
    color=[CARRIER_COLORS.get(c,'#aaaaaa') for c in nb_plot.columns])
ax.set_ylabel('GW')
ax.set_title('New Build Capacity 2030 by Scenario')
ax.legend(title='carrier', bbox_to_anchor=(1.01,1), loc='upper left', fontsize=8)
plt.tight_layout();

In [ ]:
# RE vs Fossil breakdown (total installed, not just new build)
for s in SCENARIOS:
    n = networks[s]
    cap = n.generators.groupby('carrier').p_nom_opt.sum() / 1e3
    re_gw = cap[cap.index.isin(RE_CARRIERS)].sum()
    ff_gw = cap[cap.index.isin(FOSSIL_CARRIERS)].sum()
    nuc_gw = cap.get('nuclear', 0)
    print(f'{LABELS[s]:8s}  RE={re_gw:.1f} GW  Fossil={ff_gw:.1f} GW  Nuclear={nuc_gw:.1f} GW')

## 3.2 Generation Mix 2030 [TWh]

In [ ]:
disp_all = pd.DataFrame({s: dispatch_by_carrier(networks[s]) for s in SCENARIOS}).fillna(0)
disp_all = disp_all.drop('load_shedding', errors='ignore')
disp_all = disp_all[disp_all.max(axis=1) > 0.1].round(1)
disp_all.index.name = 'carrier'
disp_all

In [ ]:
# RE fraction
for s in SCENARIOS:
    d = disp_all[s]
    total = d.sum()
    re_frac = d[d.index.isin(RE_CARRIERS)].sum() / total * 100
    coal_frac = d.get('coal', 0) / total * 100
    print(f'{LABELS[s]:8s}  total={total:.1f} TWh  RE={re_frac:.1f}%  coal={coal_frac:.1f}%')

In [ ]:
# Stacked bar chart
fig, ax = plt.subplots(figsize=(10, 6))
carriers_order = ['coal','nuclear','hydro','hydro_import','sasol_coal','sasol_gas','rmippp',
                  'ccgt_steam','solar_csp','solar_pv','solar_pv_low','solar_pv_rooftop',
                  'wind','wind_low','bioenergy']
plot_data = disp_all.T  # scenarios as rows
plot_data.index = [LABELS[s] for s in plot_data.index]
cols_present = [c for c in carriers_order if c in plot_data.columns]
plot_data[cols_present].plot.bar(
    ax=ax, stacked=True, rot=0,
    color=[CARRIER_COLORS.get(c,'#cccccc') for c in cols_present])
ax.set_ylabel('TWh')
ax.set_title('Generation Mix 2030 by Scenario')
ax.legend(title='carrier', bbox_to_anchor=(1.01,1), loc='upper left', fontsize=8)
plt.tight_layout();

## 3.3 CO₂ Emissions 2030 [MtCO₂]

> Computed from `generator_emissions.csv` (kgCO₂/MWh_el) × weighted dispatch [MWh].

In [ ]:
em_results = {s: emissions_mt(s) for s in SCENARIOS}
em_series = pd.Series(em_results, name='MtCO2')
print('CO2 emissions 2030 [MtCO2]:')
print(em_series.round(1))
print()
print('Reduction vs P0_BASE:')
print(((em_series - em_series['P0_BASE']) / em_series['P0_BASE'] * 100).round(1).astype(str) + ' %')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar([LABELS[s] for s in SCENARIOS],
              [em_results[s] for s in SCENARIOS],
              color=[COLORS[s] for s in SCENARIOS])
for bar, scen in zip(bars, SCENARIOS):
    pct = (em_results[scen] - em_results['P0_BASE']) / em_results['P0_BASE'] * 100
    label = f"{em_results[scen]:.1f}" + (f"\n({pct:+.1f}%)" if scen != 'P0_BASE' else '')
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            label, ha='center', va='bottom', fontsize=9)
ax.set_ylabel('MtCO₂')
ax.set_title('CO₂ Emissions 2030')
ax.set_ylim(0, max(em_results.values()) * 1.15)
plt.tight_layout();

In [ ]:
# Emissions by generator type (2030, all scenarios)
for s in SCENARIOS:
    n = networks[s]
    em_csv = pd.read_csv(f'{RESULTS}/{s}/outputs/generator_emissions.csv', index_col=0)
    sns, w = get_2030(n)
    gen_mwh = n.generators_t.p.loc[sns].multiply(w, axis=0).sum()
    common  = em_csv.columns.intersection(gen_mwh.index)
    ef      = em_csv.loc[2030, common]
    em_by_gen = (gen_mwh[common] * ef) / 1e9  # MtCO2 per generator
    em_by_gen.index = n.generators.loc[common, 'carrier'].values
    em_by_carrier = em_by_gen.groupby(level=0).sum().sort_values(ascending=False)
    print(f'\n{LABELS[s]}:')
    print(em_by_carrier[em_by_carrier > 0.01].round(1).to_string())

## 3.4 Carbon Tax Revenue & Reinvestment

**CT revenue** = 462 R/tCO₂ × 2030 emissions of the **reference scenario** (BASE for _R, own for CT).  
**Reinvestment** = annualised capital cost of new wind+solar built in 2030 × SCALE_COSTS (1e3).  
The constraint requires: `reinvestment ≥ CT_revenues`.

In [ ]:
# CT revenues based on each scenario's own emissions
print('CT revenues [bn ZAR] at 462 R/tCO2 × own 2030 emissions:')
for s in SCENARIOS:
    rev = ct_revenue_bn(s)
    print(f'  {LABELS[s]:8s}: {rev:.1f} bn ZAR')

In [ ]:
# Reinvestment constraint check for _R scenarios
print('Reinvestment constraint check:')
print()
for scen_r, scen_ref in [('P0_BASE_R','P0_BASE'), ('P0_CT_R','P0_CT')]:
    reinvest = reinvestment_bn(networks[scen_r])
    ct_ref = ct_revenue_bn(scen_ref)
    satisfied = '✓ BINDING' if abs(reinvest - ct_ref) / ct_ref < 0.05 else ('✓' if reinvest >= ct_ref else '✗ NOT MET')
    print(f'{LABELS[scen_r]}:')
    print(f'  Reinvestment = {reinvest:.1f} bn ZAR')
    print(f'  CT revenues (from {LABELS[scen_ref]}) = {ct_ref:.1f} bn ZAR')
    print(f'  Constraint: {satisfied}')
    print()

In [ ]:
# New build breakdown by RE_INVEST carrier for _R scenarios
for s in ['P0_BASE_R', 'P0_CT_R']:
    n = networks[s]
    re = n.generators[
        (n.generators.build_year==2030) & n.generators.carrier.isin(RE_INVEST_CARRIERS)
    ]
    print(f'\n{LABELS[s]} — RE investment carriers (2030):')
    summary = re.groupby('carrier').agg(
        p_nom_opt_GW=('p_nom_opt', lambda x: x.sum()/1e3),
        capital_cost_scaled=('capital_cost', 'mean'),
        annualised_bn=('p_nom_opt', lambda x: (x * re.loc[x.index,'capital_cost']).sum() * SCALE_COSTS / 1e9)
    ).round(2)
    print(summary)

## 3.5 System Costs

**Note on `n.objective`**: The solver minimises costs with `scale_costs(n,1e3)` applied (all costs ÷1e3). So `n.objective × 1e3` gives actual R; ÷1e9 gives bn ZAR.  
**Note on marginal costs**: `coal.marginal_cost = 0` throughout — this is a sunk-cost model. The CT is added as a marginal cost DURING solving for P0_CT/P0_CT_R but stored back at pre-CT values in the solved.nc.

In [ ]:
print('Total objective [bn ZAR] (both periods, discounted):')
for s in SCENARIOS:
    obj = networks[s].objective * SCALE_COSTS / 1e9
    print(f'  {LABELS[s]:8s}: {obj:.0f} bn ZAR')

In [ ]:
# Annual capital cost of new build 2030 [bn ZAR/yr]
print('Annualised capital cost of new build 2030 [bn ZAR/yr]:')
for s in SCENARIOS:
    n = networks[s]
    nb = n.generators[n.generators.build_year==2030]
    capex = (nb.p_nom_opt * nb.capital_cost).sum() * SCALE_COSTS / 1e9
    print(f'  {LABELS[s]:8s}: {capex:.1f} bn ZAR/yr')

In [ ]:
# Marginal (operational) costs 2030 [bn ZAR/yr]
# Note: scale_costs also divides marginal_cost by 1e3, multiply back
print('Operational costs 2030 (marginal_cost × dispatch, ×SCALE_COSTS) [bn ZAR/yr]:')
for s in SCENARIOS:
    n = networks[s]
    sns, w = get_2030(n)
    gen_2030 = n.generators_t.p.loc[sns]
    mc = get_as_dense(n, 'Generator', 'marginal_cost', sns)
    vc = (gen_2030.multiply(w, axis=0) * mc).sum().sum() * SCALE_COSTS / 1e9
    print(f'  {LABELS[s]:8s}: {vc:.1f} bn ZAR/yr')

In [ ]:
# Marginal costs by carrier [R/MWh] — time-averaged 2030, BASE vs CT
print('Marginal costs by carrier [R/MWh] — comparing BASE vs CT:')
sns_base, _ = get_2030(n_base)
sns_ct, _   = get_2030(n_ct)
mc_base_tv = get_as_dense(n_base, 'Generator', 'marginal_cost', sns_base)
mc_ct_tv   = get_as_dense(n_ct,   'Generator', 'marginal_cost', sns_ct)
mc_base = mc_base_tv.mean().groupby(n_base.generators.carrier).mean()
mc_ct   = mc_ct_tv.mean().groupby(n_ct.generators.carrier).mean()
mc_diff = pd.DataFrame({'BASE': mc_base, 'CT': mc_ct})
mc_diff['diff'] = mc_diff['CT'] - mc_diff['BASE']
mc_diff = mc_diff[mc_diff.max(axis=1) > 0]
print((mc_diff * SCALE_COSTS).round(1))  # multiply back to R/MWh

## 3.6 Summary Table — Paper

Key metrics across all 4 scenarios for 2030 (LC-182h test runs, regions=10).

In [ ]:
rows = {}
for s in SCENARIOS:
    n = networks[s]
    sns, w = get_2030(n)
    d = dispatch_by_carrier(n)
    total_gen  = d.drop('load_shedding', errors='ignore').sum()
    re_gen     = d[d.index.isin(RE_CARRIERS)].sum()
    coal_gen   = d.get('coal', 0)
    em         = emissions_mt(s)
    nb         = new_build_gw(n)
    re_new_gw  = nb[nb.index.isin(RE_CARRIERS)].sum()
    wind_new   = nb.get('wind', 0) + nb.get('wind_low', 0)
    solar_new  = nb.get('solar_pv', 0) + nb.get('solar_pv_low', 0) + nb.get('solar_pv_rooftop', 0)
    rows[LABELS[s]] = {
        'Load [TWh]':           round(total_load_twh(n), 1),
        'Total gen [TWh]':      round(total_gen, 1),
        'RE gen [TWh]':         round(re_gen, 1),
        'Coal gen [TWh]':       round(coal_gen, 1),
        'RE share [%]':         round(re_gen / total_gen * 100, 1),
        'New wind [GW]':        round(wind_new, 2),
        'New solar [GW]':       round(solar_new, 2),
        'New RE total [GW]':    round(re_new_gw, 2),
        'CO2 [MtCO2]':         round(em, 1),
        'CO2 red. vs BASE [%]': round((em - emissions_mt('P0_BASE'))/emissions_mt('P0_BASE')*100, 1),
        'CT revenue [bn ZAR]':  round(ct_revenue_bn(s), 1),
        'Reinvest [bn ZAR]':    round(reinvestment_bn(n), 1) if '_R' in s else '-',
    }

summary = pd.DataFrame(rows).T
summary

In [ ]:
# Export summary as CSV
summary.to_csv(f'{RESULTS}/paper_summary_2030_182h.csv')
print('Saved to:', f'{RESULTS}/paper_summary_2030_182h.csv')

# 4. Network Maps

In [ ]:
# Simple network plot (BASE)
try:
    fig, ax = plt.subplots(figsize=(8, 8))
    n_base.plot(ax=ax, bus_sizes=1, margin=0.1)
    ax.set_title('P0_BASE — Network Topology')
    plt.tight_layout()
except Exception as e:
    print(f'Map requires geo dependencies: {e}')

In [ ]:
# Capacity map (new build 2030, bubble size = total new RE GW per bus)
try:
    fig, axes = plt.subplots(1, 2, figsize=(18, 8),
                              subplot_kw={'projection': __import__('cartopy').crs.PlateCarree()})
    for ax, scen in zip(axes, ['P0_BASE_R', 'P0_CT_R']):
        n = networks[scen]
        new_re = n.generators[
            (n.generators.build_year==2030) & n.generators.carrier.isin(RE_CARRIERS)
        ]
        bus_sizes = new_re.groupby('bus').p_nom_opt.sum() / 5e3
        n.plot(ax=ax, bus_sizes=bus_sizes, bus_colors='green', margin=0.1)
        ax.set_title(f'{LABELS[scen]}: New RE capacity 2030 (bubble = GW/5)')
    plt.tight_layout()
except Exception as e:
    print(f'Cartopy not available or geo error: {e}')

In [ ]:
# Link loading 2030 (BASE)
sns, _ = get_2030(n_base)
if not n_base.links_t.p0.empty:
    p0 = n_base.links_t.p0.loc[sns]
    loading = p0.abs().mean() / (n_base.links.p_nom_opt.replace(0, np.nan))
    print('Mean loading of transmission links 2030 (BASE):')
    print(loading.sort_values(ascending=False).round(2).head(10))
else:
    print('No link flow data stored.')